# **Structural Damage Classification Modeling**

## Setup

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import time
from scipy import sparse
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

data_dir = "../data"
train = pd.read_csv(f"{data_dir}/train.csv")
test = pd.read_csv(f"{data_dir}/test.csv")
sample_sub = pd.read_csv(f"{data_dir}/sample_submission.csv")

train["grade"] = train.damage_grade.str[-1].astype(int)
y = train.grade.values - 1

cat_cols = ["land_surface_condition", "foundation_type", "roof_type", "ground_floor_type",
            "other_floor_type", "position", "plan_configuration", "superstructure"]
loc_cols = ["district_id", "vdcmun_id", "ward_id"]
mat_cols_5 = ["foundation_type", "roof_type", "ground_floor_type", "other_floor_type", "superstructure"]

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold = np.zeros(len(train), dtype=int)
for i, (_, vai) in enumerate(skf.split(train, y)):
    fold[vai] = i

## Base features

In [2]:
def build_features(train, test):
    tr = train.copy()
    te = test.copy()

    for df in (tr, te):
        df["age_building"] = df.age_building.clip(upper=200)
        df["age_unknown"] = (df.age_building >= 200).astype(int)
        df["height_per_floor"] = df.height_ft_pre_eq / df.count_floors_pre_eq
        df["area_per_floor"] = df.plinth_area_sq_ft / df.count_floors_pre_eq

    all_cat = cat_cols + loc_cols
    for c in all_cat:
        categories = pd.Categorical(pd.concat([tr[c].astype(str), te[c].astype(str)])).categories
        tr[c] = pd.Categorical(tr[c].astype(str), categories=categories)
        te[c] = pd.Categorical(te[c].astype(str), categories=categories)

    drop_cols = ["building_id", "damage_grade", "grade"]
    feat_cols = [c for c in tr.columns if c not in drop_cols]
    return tr[feat_cols], te[feat_cols]

X, X_test = build_features(train, test)
print(X.shape, X_test.shape)

(533465, 18) (228629, 18)


### ID-neighbor features (mean/std)
`building_id` encode geografi (first digits = district, consecutive IDs hampir selalu ward yang sama).
Untuk tiap building, mean/std grade dari K=5 nearest `building_id` row yang dilabel saja

In [3]:
def add_id_neighbor_features(train, test, fold, K=5):
    tr = train[["building_id", "grade"]].copy()
    tr["fold"] = fold
    te_ids = test[["building_id"]].copy()

    def compute(query_ids, source_df):
        src = source_df.sort_values("building_id").reset_index(drop=True)
        src_id = src.building_id.values
        src_g = src.grade.values
        out_mean = np.full(len(query_ids), np.nan)
        out_std = np.full(len(query_ids), np.nan)
        pos = np.searchsorted(src_id, query_ids.values)
        for i, (qid, p) in enumerate(zip(query_ids.values, pos)):
            lo, hi = max(0, p - K), min(len(src_id), p + K)
            idx = [j for j in range(lo, hi) if src_id[j] != qid]
            if not idx:
                continue
            dists = np.abs(src_id[np.array(idx)] - qid)
            nearest = np.array(idx)[np.argsort(dists)][:K]
            out_mean[i] = src_g[nearest].mean()
            out_std[i] = src_g[nearest].std() if len(nearest) > 1 else 0
        return out_mean, out_std

    train_mean = np.full(len(train), np.nan)
    train_std = np.full(len(train), np.nan)
    for f in range(fold.max() + 1):
        q_mask = fold == f
        source = tr[tr.fold != f][["building_id", "grade"]]
        m, s = compute(tr.loc[q_mask, "building_id"], source)
        train_mean[q_mask] = m
        train_std[q_mask] = s

    test_mean, test_std = compute(te_ids["building_id"], tr[["building_id", "grade"]])
    return (pd.DataFrame({"nbr_mean_grade": train_mean, "nbr_std_grade": train_std}),
            pd.DataFrame({"nbr_mean_grade": test_mean, "nbr_std_grade": test_std}))


nbr_train, nbr_test = add_id_neighbor_features(train, test, fold, K=5)
X[["nbr_mean_grade", "nbr_std_grade"]] = nbr_train.values
X_test[["nbr_mean_grade", "nbr_std_grade"]] = nbr_test.values
print("missing:", X[["nbr_mean_grade", "nbr_std_grade"]].isna().sum().sum())

missing: 0


### `weakest_link`: material severity
Rank tiap 5 kategori material columns menurut mean grade (fold-aware ranking).
`weakest_link` score suatu building adalah max severity rank sepanjang 5 material columnnya.
vulnerability berdasarkan weakest component buildingnya.

### `weakest_link x nbr_mean_grade`
`nbr_mean_grade` = perkiraan intensitas guncangan di area, `weakest_link` structural vulnerability buildingnya.

In [4]:
def make_weakest_link_foldaware(train, test, fold, mat_cols_5):
    train_feat = np.full(len(train), np.nan)
    for f in range(fold.max() + 1):
        tr_idx = fold != f
        va_idx = fold == f
        sev_rank_f = {c: train.loc[tr_idx].groupby(c).grade.mean().rank() for c in mat_cols_5}
        sev_scores_va = pd.DataFrame({c: train.loc[va_idx, c].map(sev_rank_f[c]) for c in mat_cols_5})
        train_feat[va_idx] = sev_scores_va.max(axis=1).values

    sev_rank_full = {c: train.groupby(c).grade.mean().rank() for c in mat_cols_5}
    sev_scores_test = pd.DataFrame({c: test[c].map(sev_rank_full[c]) for c in mat_cols_5})
    test_feat = sev_scores_test.max(axis=1).values
    return train_feat, test_feat


wl_train, wl_test = make_weakest_link_foldaware(train, test, fold, mat_cols_5)
X["weakest_link"] = wl_train
X_test["weakest_link"] = wl_test

X["weakest_link_x_nbr"] = X["weakest_link"] * X["nbr_mean_grade"]
X_test["weakest_link_x_nbr"] = X_test["weakest_link"] * X_test["nbr_mean_grade"]
print("missing:", np.isnan(wl_train).sum())

missing: 0


### ID-neighbor grade-fraction decomposition

In [5]:
def add_neighbor_distribution_features(train, test, fold, K=5):
    tr = train[["building_id", "grade"]].copy()
    tr["fold"] = fold
    te_ids = test[["building_id"]].copy()

    def compute(query_ids, source_df):
        src = source_df.sort_values("building_id").reset_index(drop=True)
        src_id = src.building_id.values
        src_g = src.grade.values
        out = np.zeros((len(query_ids), 5))
        pos = np.searchsorted(src_id, query_ids.values)
        for i, (qid, p) in enumerate(zip(query_ids.values, pos)):
            lo, hi = max(0, p - K), min(len(src_id), p + K)
            idx = [j for j in range(lo, hi) if src_id[j] != qid]
            if not idx:
                out[i] = 0.2
                continue
            dists = np.abs(src_id[np.array(idx)] - qid)
            nearest = np.array(idx)[np.argsort(dists)][:K]
            counts = np.bincount(src_g[nearest] - 1, minlength=5)
            out[i] = counts / len(nearest)
        return pd.DataFrame(out, columns=[f"nbr_frac_{g}" for g in range(1, 6)])

    train_features = pd.DataFrame(0.0, index=train.index,
                                   columns=[f"nbr_frac_{g}" for g in range(1, 6)])
    for f in range(fold.max() + 1):
        q_mask = fold == f
        source = tr[tr.fold != f][["building_id", "grade"]]
        train_features.loc[q_mask] = compute(tr.loc[q_mask, "building_id"], source).values

    test_features = compute(te_ids["building_id"], tr[["building_id", "grade"]])
    return train_features, test_features


nbr_dist_train, nbr_dist_test = add_neighbor_distribution_features(train, test, fold, K=5)
X[nbr_dist_train.columns] = nbr_dist_train.values
X_test[nbr_dist_test.columns] = nbr_dist_test.values
print("missing:", X[nbr_dist_train.columns].isna().sum().sum())

missing: 0


### Graph Label Propagation
bangunan yang `building_id` nya berdekatan biasanya berada di wilayah yang sama (mengalami tingkat kerusakan yang mirip). Label propagation membagikan informasi kerusakan ke bangunan berdekatan.

In [6]:
def build_propagation_graph(train, test, K=10):
    all_ids = pd.concat([train[["building_id"]], test[["building_id"]]], ignore_index=True)
    all_ids["orig_idx"] = range(len(all_ids))
    sorted_df = all_ids.sort_values("building_id").reset_index(drop=True)
    ids = sorted_df.building_id.values
    n = len(ids)

    rows, cols, weights = [], [], []
    for i in range(n):
        lo, hi = max(0, i - K), min(n, i + K + 1)
        for j in range(lo, hi):
            if j == i:
                continue
            dist = abs(ids[i] - ids[j])
            w = 1.0 / (dist + 1.0)
            rows.append(sorted_df.orig_idx.values[i])
            cols.append(sorted_df.orig_idx.values[j])
            weights.append(w)

    S = sparse.csr_matrix((weights, (rows, cols)), shape=(n, n))
    row_sums = np.array(S.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1
    S = sparse.diags(1.0 / row_sums) @ S
    return S


def propagate_labels(S, seed_Y, alpha=0.8, n_iter=15):
    F = seed_Y.copy()
    for _ in range(n_iter):
        F = alpha * (S @ F) + (1 - alpha) * seed_Y
    return F


def get_propagated_features(train, test, fold, S, alpha=0.8, n_iter=15):
    n_train, n_test = len(train), len(test)
    n_total = n_train + n_test
    grade_onehot = np.zeros((n_train, 5))
    grade_onehot[np.arange(n_train), train.grade.values - 1] = 1.0

    train_out = np.zeros((n_train, 5))
    for f in range(fold.max() + 1):
        seed_Y = np.zeros((n_total, 5))
        known_train_mask = fold != f
        seed_Y[:n_train][known_train_mask] = grade_onehot[known_train_mask]
        F = propagate_labels(S, seed_Y, alpha=alpha, n_iter=n_iter)
        F_norm = F / np.clip(F.sum(axis=1, keepdims=True), 1e-9, None)
        va_idx = np.where(fold == f)[0]
        train_out[va_idx] = F_norm[va_idx]

    seed_Y_test = np.zeros((n_total, 5))
    seed_Y_test[:n_train] = grade_onehot
    F_test = propagate_labels(S, seed_Y_test, alpha=alpha, n_iter=n_iter)
    F_test_norm = F_test / np.clip(F_test.sum(axis=1, keepdims=True), 1e-9, None)
    test_out = F_test_norm[n_train:]
    return train_out, test_out


print("building propagation graph...")
S = build_propagation_graph(train, test, K=10)
print("graph built:", S.shape, S.nnz, "edges")

print("running fold-aware propagation...")
prop_train, prop_test = get_propagated_features(train, test, fold, S, alpha=0.8, n_iter=15)
for i in range(5):
    X[f"prop_g{i+1}"] = prop_train[:, i]
    X_test[f"prop_g{i+1}"] = prop_test[:, i]
print("done:", X.shape, X_test.shape)

building propagation graph...
graph built: (762094, 762094) 15241770 edges
running fold-aware propagation...
done: (533465, 32) (228629, 32)


## Class weights
Inverse class frequency. Grade 5 adalah 36% dari data, Grade 1 adalah 10% dari data.

In [7]:
class_weight = (len(train) / (5 * np.bincount(y))).astype(np.float32)
w = class_weight[y]
print(class_weight.round(2))

[1.93 1.75 1.12 0.83 0.55]


## Offset tuning

In [8]:
def tune_offsets(probs, y_true, n_iter=300):
    def neg_f1(off):
        pred = (np.log(probs + 1e-9) + off).argmax(1) + 1
        return -f1_score(y_true, pred, average="macro")

    x0 = np.zeros(5)
    initial_simplex = np.vstack([x0] + [x0 + 0.5 * np.eye(5)[i] for i in range(5)])
    res = minimize(neg_f1, x0=x0, method="Nelder-Mead",
                    options={"maxiter": n_iter, "xatol": 1e-3, "fatol": 1e-6,
                             "initial_simplex": initial_simplex, "adaptive": True})
    return res.x

## Train: 5-fold LightGBM, tuned params + class weights

In [9]:
import warnings
warnings.filterwarnings("ignore", category=Warning, module="lightgbm")

lgb_params = dict(
    objective="multiclass", num_class=5, learning_rate=0.08, n_estimators=3000,
    num_leaves=127, min_child_samples=40, colsample_bytree=0.8,
    subsample=0.8, subsample_freq=1, reg_lambda=5.0, cat_smooth=20,
    cat_l2=10, n_jobs=-1, verbosity=-1, random_state=42,
)

oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))
t_start = time.time()

for f in range(N_SPLITS):
    t0 = time.time()
    tri, vai = fold != f, fold == f
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X.iloc[tri], y[tri], sample_weight=w[tri],
              eval_set=[(X.iloc[vai], y[vai])], eval_sample_weight=[w[vai]],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(period=200)])
    oof_probs[vai] = model.predict_proba(X.iloc[vai])
    test_probs += model.predict_proba(X_test) / N_SPLITS
    eta = (time.time() - t_start) / (f + 1) * (N_SPLITS - f - 1)
    print(f"[fold {f}] {time.time()-t0:.1f}s  |  stopped at iter {model.best_iteration_}  |  ETA {eta/60:.1f} min")

assert (oof_probs.sum(axis=1) == 0).sum() == 0, "some rows never got predicted -- check the loop ran fully"

[200]	valid_0's multi_logloss: 0.957145
[fold 0] 246.0s  |  stopped at iter 206  |  ETA 16.4 min
[200]	valid_0's multi_logloss: 0.958245
[fold 1] 230.9s  |  stopped at iter 178  |  ETA 11.9 min
[200]	valid_0's multi_logloss: 0.949036
[fold 2] 255.9s  |  stopped at iter 209  |  ETA 8.1 min
[200]	valid_0's multi_logloss: 0.950719
[fold 3] 302.7s  |  stopped at iter 262  |  ETA 4.3 min
[200]	valid_0's multi_logloss: 0.94975
[fold 4] 277.1s  |  stopped at iter 235  |  ETA 0.0 min


## Evaluate + tune offsets

In [10]:
raw_pred = oof_probs.argmax(1) + 1
print("raw argmax OOF macro F1:", f1_score(train.grade.values, raw_pred, average="macro"))

off = tune_offsets(oof_probs, train.grade.values)
tuned_pred = (np.log(oof_probs + 1e-9) + off).argmax(1) + 1
print("tuned OOF macro F1:", f1_score(train.grade.values, tuned_pred, average="macro"))

raw argmax OOF macro F1: 0.5863540132827588
tuned OOF macro F1: 0.5920578434002023


## Save

In [12]:
np.save("../models/test_probs.npy", test_probs)
np.save("../models/offsets.npy", off)